In [97]:
import json
from collections import defaultdict
from educast.constants import INTERIM_DATA_DIR
from educast.data.models import University

EPSEM_EDUCAST_PATH = INTERIM_DATA_DIR / "EPSEM_data_private" / "EPSEM.educast.json"

# Load EduCast data
with open(EPSEM_EDUCAST_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

university = University.model_validate(data)

In [98]:
print(f"University: {university.name}")
print(f"Total Students: {len(university.students)}")
print(f"Total Enrollments: {university.total_enrollments}")

first_student = university.students[0]
print()
print(f"First Student ID: {first_student.student_id}")
print(f"Enrollments: {first_student.enrollments}")
print(f"Attempts: {first_student.attempts}")
print(f"Student GPA: {first_student.history.gpa}")
print(f"Student completed credits: {first_student.history.total_credits}")

University: EPSEM (Escola Politècnica Superior d'Enginyeria de Manresa)
Total Students: 505
Total Enrollments: 2663

First Student ID: 75745a
Enrollments: 8
Attempts: 37
Student GPA: 8.11
Student completed credits: 240.0


In [99]:
from collections import defaultdict

sequences = []

for student in university.students:
    term_dict = defaultdict(list)
    for attempt in student.history.attempts:
        key = (attempt.year, attempt.term)
        term_dict[key].append(attempt.course.course_id)
    
    # Sort by chronological order
    student_sequence = [term_dict[k] for k in sorted(term_dict.keys())]
    sequences.append(student_sequence)

In [100]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [102]:
from collections import defaultdict

sequences = []

for student in university.students:
    term_dict = defaultdict(list)
    for attempt in student.history.attempts:
        key = (attempt.year, attempt.term)
        term_dict[key].append(attempt.course.course_id)
    
    # Sort by chronological order
    student_sequence = [term_dict[k] for k in sorted(term_dict.keys())]
    sequences.append(student_sequence)

for i, seq in enumerate(sequences[:3]):
    print(f"\nStudent {i+1} course sequence by term:")
    for term_courses in seq:
        print(term_courses)
print(sequences[0])

# Maximum number of terms any student has (i.e., longest enrollment in terms)
max_terms = max((len(seq) for seq in sequences), default=0)
students_with_max = [i for i, seq in enumerate(sequences) if len(seq) == max_terms]

# Maximum number of courses in a single term (enrollment) across all students
max_courses = 0
students_with_max_courses = []  # list of (student_index, term_index)

for i, seq in enumerate(sequences):
    for term_idx, term_courses in enumerate(seq):
        n = len(term_courses)
        if n > max_courses:
            max_courses = n
            students_with_max_courses = [(i, term_idx)]
        elif n == max_courses:
            students_with_max_courses.append((i, term_idx))


Student 1 course sequence by term:
['330212', '330213', '330214', '330215', '330216']
['330217', '330218', '330219', '330220', '330221']
['330222', '330223', '330224', '330225', '330226']
['330227', '330228', '330229', '330230', '330231']
['330232', '330233', '330234', '330235', '330236']
['330237', '330238', '330239', '330240', '330248']
['330241', '330242', '330246', '330247']
['330094', '330102', '330243']

Student 2 course sequence by term:
['330212', '330213', '330214', '330215', '330216']
['330217', '330218', '330219', '330220', '330221']
['330099', '330213', '330222', '330223', '330224', '330225', '330226']
['330228', '330229']
['330222', '330226', '330233', '330235']
['330227', '330230', '330231']
['330232', '330234', '330236', '330242']
['330237', '330238', '330239', '330240', '330248']
['330241', '330246']
['330243', '330247']

Student 3 course sequence by term:
['330212', '330213', '330214', '330215', '330216']
['330217', '330218', '330219', '330220', '330221']
['330099', '

In [103]:
import numpy as np

# Create a mapping from course_id to index
all_course_ids = [c.course_id for c in university.programmes[0].courses]
course_to_idx = {cid: i for i, cid in enumerate(all_course_ids)}
num_courses = len(all_course_ids)

# Convert sequences to multi-hot vectors per term
encoded_sequences = []

for seq in sequences:
    encoded_seq = []
    for term_courses in seq:
        vec = np.zeros(num_courses, dtype=int)
        for cid in term_courses:
            if cid in course_to_idx:
                vec[course_to_idx[cid]] = 1
        encoded_seq.append(vec)
    encoded_sequences.append(encoded_seq)

In [104]:
print(len(encoded_sequences[0]), len(encoded_sequences[0][0]))
encoded_sequences[0]

8 51


[array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
        1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 1, 1, 1, 0,

In [54]:
print(len(encoded_sequences[1]), len(encoded_sequences[1][0]))
encoded_sequences[1]

10 51


[array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

encoded_sequences_2d = [np.array(s) for s in encoded_sequences]

# We are padding in the term axis (first dimension)
padded_sequences = pad_sequences(
    encoded_sequences_2d,
    padding="post",
    dtype="int32"
)  # shape: (num_students, max_terms, num_courses)

array([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
        1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 1, 1, 1, 0, 0, 0

In [105]:
X_list = []
y_list = []

for student_seq in encoded_sequences:
    n_terms = len(student_seq)
    if n_terms < 2:
        continue  # nothing to predict
    for i in range(1, n_terms):
        X_list.append(student_seq[:i])
        y_list.append(student_seq[i])

from tensorflow.keras.preprocessing.sequence import pad_sequences

X_padded = pad_sequences(
    X_list,
    padding="post",
    dtype="int32"
)

y_array = np.array(y_list, dtype="int32")


In [106]:
X_padded.shape, y_array.shape

((2158, 18, 51), (2158, 51))

In [107]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Take just the first student
seq = encoded_sequences[0]  

X_list = []
y_list = []

n_terms = len(seq)
if n_terms < 2:
    print("Not enough terms to predict next course")
else:
    for i in range(1, n_terms):
        X_list.append(seq[:i])   # all previous terms
        y_list.append(seq[i])    # next term to predict

# Pad X so all sequences have the same number of terms
X_padded = pad_sequences(
    X_list,
    padding="post",
    dtype="int32"
)

y_array = np.array(y_list, dtype="int32")

print("X_padded shape:", X_padded.shape)
print("y_array shape:", y_array.shape)
print("\nExample X[0]:\n", X_padded[0])
print("\nExample y[0]:\n", y_array[0])

X_padded shape: (7, 7, 51)
y_array shape: (7, 51)

Example X[0]:
 [[1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]

Example y[0]:
 [0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [108]:
print("\nExample X[1]:\n", X_padded[1])
print("\nExample y[1]:\n", y_array[1])


Example X[1]:
 [[1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]

Example y[1]:
 [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [109]:
X_list = []
y_list = []

for student_seq in encoded_sequences:
    n_terms = len(student_seq)
    if n_terms < 2:
        continue
    for i in range(1, n_terms):
        X_list.append(student_seq[:i])
        y_list.append(student_seq[i])

from tensorflow.keras.preprocessing.sequence import pad_sequences

X_padded = pad_sequences(
    X_list,
    padding="post",  # 0's at the end
    dtype="int32"
)

y_array = np.array(y_list, dtype="int32")

print("\nExample X[1]:\n", X_padded[0])
print("\nExample y[1]:\n", y_array[0])


Example X[1]:
 [[1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0

In [110]:
print("X shape:", X_padded.shape)
print("y shape:", y_array.shape)

X shape: (2158, 18, 51)
y shape: (2158, 51)


In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss, jaccard_score

# Build LSTM model
model = Sequential([
    Masking(mask_value=0.0, input_shape=(X_padded.shape[1], X_padded.shape[2])), # ignores vectores with all 0's (no backprop and updades)
    LSTM(128, return_sequences=False),
    Dense(y_array.shape[1], activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = model.fit(X_padded, y_array, epochs=10, batch_size=32, validation_split=0.1)

y_pred_prob = model.predict(X_padded)
y_pred = (y_pred_prob > 0.5).astype(int)

f1 = f1_score(y_array, y_pred, average="micro")
hamming = hamming_loss(y_array, y_pred)
jaccard = jaccard_score(y_array, y_pred, average="samples")

print(f"Micro F1-score: {f1:.4f}")
print(f"Hamming loss: {hamming:.4f}")
print(f"Jaccard score: {jaccard:.4f}")

# Example: predict next term for first test student
print("Predicted next term multi-hot:", y_pred[0])

Epoch 1/10


/Users/anassanhari/EPSEM/TFM/multimodal-hierarchical-forecasting/env/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


61/61 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.0139 - loss: 0.4211 - val_accuracy: 0.1667 - val_loss: 0.3782
Epoch 2/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.1226 - loss: 0.2524 - val_accuracy: 0.5185 - val_loss: 0.2586
Epoch 3/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.3491 - loss: 0.1859 - val_accuracy: 0.5787 - val_loss: 0.1668
Epoch 4/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3615 - loss: 0.1339 - val_accuracy: 0.5231 - val_loss: 0.1124
Epoch 5/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3172 - loss: 0.1095 - val_accuracy: 0.2083 - val_loss: 0.0921
Epoch 6/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3378 - loss: 0.1003 - val_accuracy: 0.0324 - val_loss: 0.0831
Epoch 7/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2255 - loss: 0.0954 - val_accuracy: 0.2176 - val_loss: 0.0790
Epoch 8/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2868 - loss: 0.0926 - val_accuracy: 0.2130 - val_loss: 0.

In [112]:
y_pred[1]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0])

In [113]:
y_pred[2]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0])